# Sábado: responder un rango de las 992 preguntas en Kaggle (GPU) → `submissions.jsonl`

Solo genera respuestas; no evalúa (no hay claves) y no necesita la llave del juez.

**Preparación:**
1. Cuenta de Kaggle **verificada con teléfono** (sin eso no hay GPU ni internet).
2. *Datasets → New Dataset*, privado, con **dos archivos**: `paquete_colab_muestra.zip` y el archivo
   de preguntas del sábado con un nombre que empiece por **`test_992`** (p. ej. `test_992.jsonl`).
3. *Code → New Notebook → File → Import Notebook* → este archivo.
4. Panel derecho: *Accelerator* → **GPU T4 x2**; *Internet* → **On**; *Input → Add Input* → el Dataset.
5. En la **celda 2** poner `TAG` y `RANGO` de esta máquina. Luego *Run All*.

Al final aparece el enlace a `<TAG>_submissions.jsonl` (también en el panel *Output*). Si la celda 5
se interrumpe pero la sesión sigue viva, vuelva a ejecutar **solo la celda 5**: continúa donde quedó.
Si la sesión se reinicia, *Run All* empieza el rango desde cero.

In [ ]:
# 1. Comprobar GPU (deben aparecer las T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Parámetros de esta máquina y paquete

In [ ]:
import glob, os
# ===== AJUSTAR EN CADA MÁQUINA =====
TAG = "sabado_1"      # sabado_1 … sabado_6: uno distinto por máquina
RANGO = "1 165"       # posiciones del archivo de preguntas, ambos extremos incluidos
# ===================================
SPLIT = "test"
assert RANGO.strip(), "RANGO no puede quedar vacío"
INICIO, FIN = map(int, RANGO.split())

# El Dataset aparece en /kaggle/input/<nombre>/ (solo lectura), como zip o ya descomprimido.
# Si hay varios paquetes en Input, se usa el más reciente según su PAQUETE.txt (fecha de
# generación); los paquetes viejos sin PAQUETE.txt quedan al final. Lo ideal: uno solo en Input.
def _fecha(raiz):
    f = os.path.join(raiz, "PAQUETE.txt")
    return open(f, encoding="utf-8").readline().strip() if os.path.isfile(f) else ""

!rm -rf /kaggle/working/proyecto && mkdir -p /kaggle/working/proyecto
raices = [os.path.dirname(os.path.dirname(p))
          for p in glob.glob("/kaggle/input/**/data/sample_50.jsonl", recursive=True)]
zips = glob.glob("/kaggle/input/**/paquete_colab_muestra*.zip", recursive=True)
if raices:
    raices.sort(key=_fecha, reverse=True)
    for r in raices:
        print("Paquete en Input:", r, "→", _fecha(r) or "sin PAQUETE.txt (viejo)")
    print("Se usa:", raices[0])
    !cp -r "{raices[0]}"/. /kaggle/working/proyecto/
elif zips:
    print("Zip encontrado:", zips[0])
    !unzip -q "{zips[0]}" -d /kaggle/working/proyecto
else:
    raise SystemExit("No encuentro el paquete: agregue el Dataset con paquete_colab_muestra.zip en Input (panel derecho).")
%cd /kaggle/working/proyecto
!ls -la build/indice build/cache/emb data
!cat PAQUETE.txt   # debe coincidir con lo que imprimió paquete_colab al generar el zip

## 3. Dependencias (≈ 3–5 min)

In [ ]:
!pip install -q sentence-transformers==6.1.0 transformers==5.17.0 faiss-cpu==1.15.1 bm25s==0.3.11 PyStemmer==3.1.0 python-dotenv==1.2.3 jsonschema==4.26.0 httpx huggingface_hub
!pip install -q llama-cpp-python==0.3.35 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 --only-binary llama-cpp-python || echo "SIN RUEDA PRECOMPILADA: ejecute la celda 3b"

In [ ]:
# 3b. Solo compila (≈ 15 min) si no quedó instalada la rueda precompilada de llama-cpp-python.
import subprocess, sys
if subprocess.run([sys.executable, "-m", "pip", "show", "llama-cpp-python"], capture_output=True).returncode != 0:
    !CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python==0.3.35 --no-binary llama-cpp-python --force-reinstall --no-deps
else:
    print("llama-cpp-python ya instalado: no se compila.")

In [ ]:
# 3c. Verificar que llama.cpp ve la GPU (debe decir True).
# La rueda precompilada de llama.cpp es para CUDA 12. Si la imagen del entorno trae otra versión
# (p. ej. Colab con CUDA 13: "libcudart.so.12: cannot open shared object file"), se instalan las
# librerías de CUDA 12 como paquetes de Python y se apunta a ellas; los !python heredan la ruta.
import ctypes, glob, os, subprocess, sys
prueba = subprocess.run([sys.executable, "-c", "import llama_cpp"], capture_output=True, text=True)
if prueba.returncode != 0 and ".so.12" in prueba.stderr:
    print("Faltan librerías de CUDA 12: instalándolas…")
    !pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
    libs = [d for pat in ("nvidia/cuda_runtime/lib", "nvidia/cublas/lib")
            for d in glob.glob(f"/usr/local/lib/python3*/dist-packages/{pat}")]
    os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
    for d in libs:
        for so in sorted(glob.glob(d + "/lib*.so.12")):
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
import llama_cpp
print("GPU disponible para llama.cpp:", llama_cpp.llama_supports_gpu_offload())

## 4. GPU, índice y prueba del decoder (debe terminar con `CÓDIGO DE SALIDA: 0`)

In [ ]:
# por si otra celda cambió de carpeta (permite reejecutar en cualquier orden)
%cd /kaggle/working/proyecto
import os
# Decoder: por defecto Gemma 4 E4B en GGUF Q8_0 (C-15). Para Qwen3-8B: "Qwen/Qwen3-8B-GGUF"
# y "Qwen3-8B-Q4_K_M.gguf". Para probar otro modelo abierto (≤ 8B):
# agregarlo a MODELOS_ABIERTOS en src/config.py con su licencia y cambiar estas dos líneas
# (repositorio y archivo .gguf de Hugging Face). Ver docs/COMO_EJECUTAR.md §10.
DECODER_GGUF_REPO = "ggml-org/gemma-4-E4B-it-GGUF"
DECODER_GGUF_FILE = "gemma-4-E4B-it-Q8_0.gguf"
os.environ.update({"DECODER_GGUF_REPO": DECODER_GGUF_REPO, "DECODER_GGUF_FILE": DECODER_GGUF_FILE})
os.environ.update({"ENCODER_DEVICE": "cuda", "ENCODER_BATCH_SIZE": "64",
                   "DECODER_DEVICE": "cuda", "DECODER_BACKEND": "llamacpp",
                   "CORPUS_SOURCE": "local", "PYTHONIOENCODING": "utf-8",
                   # Kaggle trae dos T4: se usa una, como en Colab, para que los tiempos sean comparables.
                   "CUDA_VISIBLE_DEVICES": "0",
                   # JAX/TF preinstalados reservan el 75 % de la VRAM si se importan: a CPU.
                   "JAX_PLATFORMS": "cpu", "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
                   "TF_FORCE_GPU_ALLOW_GROWTH": "true", "USE_TF": "0", "USE_JAX": "0"})
!python -m src.index.build; echo "CÓDIGO DE SALIDA: $? (0 = bien)"
!cat build/indice/index_manifest.json

In [ ]:
# 4b. Probar el decoder en la GPU
# por si otra celda cambió de carpeta (permite reejecutar en cualquier orden)
%cd /kaggle/working/proyecto
!python -m src.generation.ping --sin-cache
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 5. Responder el rango

In [ ]:
# por si otra celda cambió de carpeta (permite reejecutar en cualquier orden)
%cd /kaggle/working/proyecto
import glob, os, shutil
if SPLIT == "test":
    _q = sorted(glob.glob("/kaggle/input/**/test_992*.jsonl", recursive=True),
                key=os.path.getmtime, reverse=True)
    if not _q:
        raise SystemExit("No encuentro test_992*.jsonl en Input: súbalo primero.")
    os.makedirs("data", exist_ok=True)
    shutil.copy(_q[0], "data/test_992.jsonl")
    print("Preguntas:", _q[0], "→", sum(1 for l in open("data/test_992.jsonl", encoding="utf-8") if l.strip()))
!python -m src.pipeline.main --split {SPLIT} --tag {TAG} --rango {RANGO}; echo "CÓDIGO DE SALIDA: $? (0 = bien; cualquier otro número = error, revise arriba)"
!cat runs/{TAG}/tiempos.json

## 6. Entregar `submissions.jsonl`

In [ ]:
import json, shutil
%cd /kaggle/working/proyecto
origen = f"runs/{TAG}/submissions.jsonl"
filas = [json.loads(l) for l in open(origen, encoding="utf-8") if l.strip()]
total = sum(1 for l in open("data/test_992.jsonl", encoding="utf-8") if l.strip())
esperadas = min(FIN, total) - INICIO + 1
print(f"{len(filas)} respuestas (se esperaban {esperadas}); ids {filas[0]['id']} … {filas[-1]['id']}")
if len(filas) != esperadas:
    print("ATENCIÓN: faltan respuestas; vuelva a ejecutar la celda 5 (continúa donde quedó).")
destino = f"/kaggle/working/{TAG}_submissions.jsonl"
shutil.copy(origen, destino)
%cd /kaggle/working
from IPython.display import FileLink
FileLink(f"{TAG}_submissions.jsonl")